# Stage 13 Homework Starter — Productization

## Objective
Deploy your trained model as a **reusable, handoff-ready API or dashboard** and finalize your project for reproducibility and clarity.

## Steps
1. Create a mock, very basic analysis in a notebook.
2. Clean your notebook by removing exploratory cells and documenting your code.
3. Move reusable functions into `/src/`.
4. Load your trained model from Stage 12 or earlier stages.
5. Pickle/save the model and test reload.
6. Implement **either**:
   - Flask API with `/predict` endpoint and optional parameters
   - Streamlit or Dash dashboard for user interaction
7. Include:
   - Error handling for invalid inputs
   - `requirements.txt` for reproducibility
   - Documentation in `README.md`
8. Test your deployment locally and provide evidence.
9. Organize project folders and finalize notebooks for handoff.

## 1. Create mock, very basic analysis

In [ ]:
# TODO: Basic analysis step 1
# TODO: Basic analysis step 2
# TODO: ...
print("Basic analysis complete.")

## 2. Notebook Cleanup
Remove exploratory cells and document your code.

In [ ]:
# TODO: Remove exploratory cells
# TODO: Document your code clearly
# Example placeholder for cleaned analysis
print("Notebook cleaned and ready for handoff.")

## 3. Move reusable functions to /src/
Create src/utils.py and store functions there.

In [ ]:
# TODO: Move actual reusable functions here
def calculate_metrics(df):
    return df.describe()

## 4. Folder Structure Reminder

Ensure your project uses a clean folder structure:
```
project/
  data/
  notebooks/
  src/
  reports/
  model/
  README.md
```
For API/Dashboard: minimal example:
```
project/
    app.py
    model.pkl
    requirements.txt
    README.md
```

## 5. Pickle / Save Final Model

### TODO: Replace this with your trained model

In [ ]:
import pickle
# TODO: Replace 'model' with your trained model variable
with open('model/model.pkl', 'wb') as f:
    pickle.dump(model, f)

# TODO: Test loading the model
with open('model/model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Example prediction (replace with actual features)
print(loaded_model.predict([[0.1, 0.2]]))

In [ ]:
# Final Analysis - SPY Returns Prediction
# This notebook demonstrates the trained linear regression model for SPY returns prediction

import yfinance as yf
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

import os
import sys 

# Get the correct paths
current_dir = os.getcwd()
bootcamp_root = os.path.dirname(current_dir) 

# Add bootcamp root to Python path to import src.cleaning
if bootcamp_root not in sys.path:
    sys.path.append(bootcamp_root)

# Add src directory specifically
src_path = os.path.join(bootcamp_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from src.utils import prepare_features, calculate_metrics

# Load data
print("Loading SPY data from Yahoo Finance...")
ticker = "SPY"
data = yf.download(ticker, start="2015-01-01", end="2023-12-31", progress=False)

# Prepare features
clean_data = prepare_features(data)
X = clean_data[['Volume_MA', 'SMA_10', 'SMA_50', 'RSI', 'MACD']]
y = clean_data['Returns']

print(f"Clean dataset shape: {clean_data.shape}")
print(f"Features: {list(X.columns)}")

In [ ]:
# Split data chronologically
split_idx = int(0.8 * len(X))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluate model
metrics = calculate_metrics(y_test, y_pred)
print(f"R² Score: {metrics['r2_score']:.4f}")
print(f"RMSE: {metrics['rmse']:.6f}")
print(f"Coefficients: {dict(zip(X.columns, model.coef_))}")

## 6. Flask API Starter

### TODO: Implement Flask endpoints for /predict and /plot

In [ ]:
from flask import Flask, request, jsonify
import threading
import matplotlib.pyplot as plt
import io
import base64

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    # TODO: Replace placeholder logic with actual model prediction
    data = request.get_json()
    features = data.get('features', None)
    if features is None:
        return jsonify({'error': 'No features provided'}), 400
    pred = sum(features)  # placeholder
    return jsonify({'prediction': pred})

@app.route('/predict/<float:input1>', methods=['GET'])
def predict_one(input1):
    pred = input1 * 2  # placeholder
    return jsonify({'prediction': pred})

@app.route('/predict/<float:input1>/<float:input2>', methods=['GET'])
def predict_two(input1, input2):
    pred = input1 + input2  # placeholder
    return jsonify({'prediction': pred})

@app.route('/plot')
def plot():
    # TODO: Replace with meaningful chart or image
    fig, ax = plt.subplots()
    ax.plot([0, 1, 2], [0, 1, 4])
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    img_bytes = base64.b64encode(buf.read()).decode('utf-8')
    return f'<img src="data:image/png;base64,{img_bytes}"/>'

def run_flask():
    app.run(port=5000)

# Launch Flask in a separate thread
threading.Thread(target=run_flask).start()

In [ ]:
from flask import Flask, request, jsonify
import pickle
import numpy as np
import pandas as pd
import matplotlib
# Use non-interactive backend to avoid threading issues
matplotlib.use('Agg')  # Must be set before importing pyplot
import matplotlib.pyplot as plt
import io
import base64
from src.utils import prepare_features

app = Flask(__name__)

# Load model and scaler
try:
    with open('model/model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('model/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    print("Model and scaler loaded successfully!")
except FileNotFoundError:
    print("Model files not found. Please train the model first.")
    model = None
    scaler = None

@app.route('/predict', methods=['POST'])
def predict():
    """Predict returns from JSON features"""
    if model is None or scaler is None:
        return jsonify({'error': 'Model not loaded'}), 500
        
    try:
        data = request.get_json()
        if not data or 'features' not in data:
            return jsonify({'error': 'No features provided'}), 400
            
        features = data['features']
        if len(features) != 5:
            return jsonify({'error': 'Expected 5 features: [Volume_MA, SMA_10, SMA_50, RSI, MACD]'}), 400
            
        # Scale features and predict
        features_scaled = scaler.transform([features])
        prediction = model.predict(features_scaled)[0]
        
        return jsonify({
            'prediction': float(prediction),
            'features': features,
            'message': 'SPY returns prediction'
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/predict/<float:input1>', methods=['GET'])
def predict_one(input1):
    """Predict returns using only Volume_MA (single feature)"""
    if model is None or scaler is None:
        return jsonify({'error': 'Model not loaded'}), 500
        
    try:
        # Use default values for other features
        default_features = [300.0, 290.0, 50.0, 0.0]  # SMA_10, SMA_50, RSI, MACD
        
        features = [input1] + default_features
        features_scaled = scaler.transform([features])
        prediction = model.predict(features_scaled)[0]
        
        return jsonify({
            'prediction': float(prediction),
            'input_used': {'Volume_MA': input1},
            'default_features_used': {
                'SMA_10': default_features[0],
                'SMA_50': default_features[1],
                'RSI': default_features[2],
                'MACD': default_features[3]
            },
            'message': 'Prediction using Volume_MA with default values for other features'
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 400
    
@app.route('/predict/<float:input1>/<float:input2>', methods=['GET'])
def predict_two(input1, input2):
    """Predict returns using Volume_MA and RSI (two features)"""
    if model is None or scaler is None:
        return jsonify({'error': 'Model not loaded'}), 500
        
    try:
        # Use default values for remaining features
        default_features = [300.0, 290.0, 0.0]  # SMA_10, SMA_50, MACD
        
        features = [input1, default_features[0], default_features[1], input2, default_features[2]]
        features_scaled = scaler.transform([features])
        prediction = model.predict(features_scaled)[0]
        
        return jsonify({
            'prediction': float(prediction),
            'inputs_used': {'Volume_MA': input1, 'RSI': input2},
            'default_features_used': {
                'SMA_10': default_features[0],
                'SMA_50': default_features[1],
                'MACD': default_features[2]
            },
            'message': 'Prediction using Volume_MA and RSI with default values for other features'
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/plot')
def plot():
    """Generate sample plot of model coefficients"""
    if model is None:
        return jsonify({'error': 'Model not loaded'}), 500
        
    try:
        # Create bar plot of coefficients
        feature_names = ['Volume_MA', 'SMA_10', 'SMA_50', 'RSI', 'MACD']
        coefficients = model.coef_
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.bar(feature_names, coefficients)
        ax.set_title('Linear Regression Coefficients')
        ax.set_ylabel('Coefficient Value')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        
        # Convert to base64
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img_bytes = base64.b64encode(buf.read()).decode('utf-8')
        plt.close(fig)
        
        return f'<img src="data:image/png;base64,{img_bytes}" alt="Model Coefficients"/>'
        
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/health')
def health():
    """Health check endpoint"""
    return jsonify({
        'status': 'healthy',
        'model_loaded': model is not None,
        'scaler_loaded': scaler is not None
    })

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)

## 7. Testing the Flask API from Notebook

### TODO: Modify examples with your actual features

In [ ]:
import requests
from IPython.display import display, HTML

# POST /predict
response = requests.post(
    'http://127.0.0.1:5000/predict',
    json={'features':[0.1, 0.2, 0.3]}
)
print(response.json())

# GET /predict/<input1>
response2 = requests.get('http://127.0.0.1:5000/predict/2.0')
print(response2.json())

# GET /predict/<input1>/<input2>
response3 = requests.get('http://127.0.0.1:5000/predict/1.0/3.0')
print(response3.json())

# GET /plot
response_plot = requests.get('http://127.0.0.1:5000/plot')
display(HTML(response_plot.text))

In [ ]:
# Test the Flask API - CORRECTED VERSION
import requests
import json
from IPython.display import display

# Test POST /predict (this works)
response = requests.post(
    'http://127.0.0.1:5000/predict',
    json={'features': [1000000, 450.0, 440.0, 55.0, 0.5]}
)
print("POST /predict response:")
print(response.json())

# Test GET /predict/<input1> (single feature - Volume_MA)
print("Testing GET /predict/<input1> (Volume_MA only):")
response2 = requests.get('http://127.0.0.1:5000/predict/1500000')
print(json.dumps(response2.json(), indent=2))
print()

# Test GET /predict/<input1>/<input2> (two features - Volume_MA and RSI)
print("Testing GET /predict/<input1>/<input2> (Volume_MA and RSI):")
response3 = requests.get('http://127.0.0.1:5000/predict/1200000/60.0')
print(json.dumps(response3.json(), indent=2))
print()

# Test /plot - DON'T use .json() here!
response_plot = requests.get('http://127.0.0.1:5000/plot')
print("\nPlot endpoint status:", response_plot.status_code)
print("Plot content type:", response_plot.headers.get('content-type'))

# Display the plot properly
if response_plot.status_code == 200:
    display(HTML(response_plot.text))
else:
    print("Plot request failed:", response_plot.text)

# Test health endpoint
health_response = requests.get('http://127.0.0.1:5000/health')
print("\nHealth check:")
print(health_response.json())

## 8. Optional Streamlit / Dash Dashboard

### TODO: Add dashboard in a separate file (`app_streamlit.py` or `app_dash.py`)

## 9. Handoff Best Practices

- Ensure README.md is complete and clear
- Provide `requirements.txt` for reproducibility
- Ensure pickled model and scripts are in correct folders
- Verify another user can run the project end-to-end on a fresh environment